# Synthesis + Performance Extraction Pipeline

This notebook demonstrates the step-by-step extraction of:
1. **Materials** from a scientific paper
2. **Synthesis procedures** for each material
3. **Performance data** from plots, linked to materials

Each step produces visible output so you can inspect intermediate results.

## Setup: Configuration

In [2]:
# ==============================================================================
# USER CONFIGURATION - Edit these values
# ==============================================================================

# Path to your PDF or markdown file
INPUT_PATH = "/Users/valeriegentzke/Documents/Studieren/lemat_synth/lematerial-llm-synthesis/examples/data/pdf_papers/04_Wu_2020_Ammonia.pdf"  # or .md file with embedded images


# Output directory for results
OUTPUT_DIR = "results/"

# Models to use
GEMINI_MODEL = "gemini-2.0-flash"  # For synthesis extraction
CLAUDE_MODEL = "claude-sonnet-4-20250514"  # For plot data extraction
LINKER_MODEL = "gemini-3-pro-preview"  # For series-to-material matching

# Set to True to skip figure/performance extraction (synthesis only)
SKIP_FIGURES = False

In [3]:
# Load environment and imports
import os
import sys
import json
import warnings
from pathlib import Path

# Add src directory to Python path (required for imports)
src_path = Path("../../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from dotenv import load_dotenv

# Load .env file
env_path = Path("../../.env")  # Adjust path if needed
load_dotenv(env_path, override=True)

# Silence noisy loggers
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

import logging
logging.getLogger("pydantic").setLevel(logging.ERROR)
logging.getLogger("LiteLLM").setLevel(logging.ERROR)
logging.getLogger("litellm").setLevel(logging.ERROR)

print("[OK] Environment loaded")
print(f"[OK] src path added: {src_path}")

[OK] Environment loaded
[OK] src path added: /Users/valeriegentzke/Documents/Studieren/lemat_synth/lematerial-llm-synthesis/src


## Step 0: Load Paper Text

If you have a PDF, we first extract text using Docling. If you already have markdown, we load it directly.

In [4]:
from llm_synthesis.data_loader.paper_loader.fs_paper_loader import FSPaperLoader
from llm_synthesis.models.paper import Paper

input_path = Path(INPUT_PATH)

if input_path.suffix.lower() == ".pdf":
    # Extract text from PDF
    print(f"Extracting text from PDF: {input_path.name}")
    from llm_synthesis.transformers.pdf_extraction import DoclingPDFExtractor
    
    extractor = DoclingPDFExtractor()
    with open(input_path, "rb") as f:
        paper_text = extractor.forward(f.read())
    
    paper = Paper(
        name=input_path.stem,
        id=input_path.stem,
        publication_text=paper_text,
        si_text="",
    )
    print(f"   Extracted {len(paper_text):,} characters")
    
elif input_path.suffix.lower() in [".md", ".txt"]:
    # Load markdown directly
    print(f"Loading markdown: {input_path.name}")
    with open(input_path, "r", errors="replace") as f:
        paper_text = f.read()
    
    paper = Paper(
        name=input_path.stem,
        id=input_path.stem,
        publication_text=paper_text,
        si_text="",
    )
    print(f"   Loaded {len(paper_text):,} characters")
    
elif input_path.is_dir():
    # Load from directory using FSPaperLoader
    print(f"Loading papers from directory: {input_path}")
    loader = FSPaperLoader(data_dir=str(input_path))
    papers = loader.load()
    paper = papers[0]  # Use first paper
    print(f"   Loaded paper: {paper.name} ({len(paper.publication_text):,} characters)")
else:
    raise ValueError(f"Unsupported input type: {input_path}")

print("\n[OK] Paper loaded successfully")
print(f"   Paper ID: {paper.id}")
print(f"   Paper Name: {paper.name}")

Extracting text from PDF: 04_Wu_2020_Ammonia.pdf


/Users/valeriegentzke/Documents/Studieren/lemat_synth/lematerial-llm-synthesis/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/valeriegentzke/Documents/Studieren/lemat_synth/lematerial-llm-synthesis/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


   Extracted 1,279,056 characters

[OK] Paper loaded successfully
   Paper ID: 04_Wu_2020_Ammonia
   Paper Name: 04_Wu_2020_Ammonia


In [5]:
# Preview paper text
print("=" * 60)
print("PAPER TEXT PREVIEW (first 2000 chars)")
print("=" * 60)
print(paper.publication_text[:2000])
print("...")

PAPER TEXT PREVIEW (first 2000 chars)
i n t e r n a t i o n a l j o u r n a l of hydrogen energy xxx (xxxx) xxx

![Image](
...


## Step 1: Extract Materials

Identify all materials that were synthesized in this paper.

In [6]:
from llm_synthesis.transformers.material_extraction.dspy_extraction import (
    DspyTextExtractor,
    make_dspy_text_extractor_signature,
)
from llm_synthesis.utils.dspy_utils import get_llm_from_name
from llm_synthesis.utils import clean_text

# Create material extractor
material_sig = make_dspy_text_extractor_signature(
    instructions=(
        "Extract ALL distinct material compositions that were synthesized and tested in this paper. "
        "IMPORTANT: If the paper studies multiple variants of a material (e.g., different loadings, "
        "dopant concentrations, or preparation conditions), list EACH variant as a separate material. "
        "For example, if a paper studies 1%Ru/CaO, 3%Ru/CaO, and 5%Ru/CaO, list all three - "
        "do NOT merge them into a single 'Ru/CaO'. "
        "Focus on materials that were actually synthesized, not just mentioned or referenced."
    ),
    output_description=(
        "ALL distinct synthesized material compositions as a comma-separated list using chemical formulas. "
        "Include loading percentages and promoters when specified "
        "(e.g., '1%Ru-10%K/CaO, 3%Ru-10%K/CaO, 5%Ru-10%K/CaO, 3%Ru-5%K/CaO'). "
        "Never merge variants into a single generic name."
    ),
)

material_lm = get_llm_from_name(
    "gemini-2.5-flash-lite",
    model_kwargs={"temperature": 0.0},
)
material_extractor = DspyTextExtractor(signature=material_sig, lm=material_lm)

print("Extracting materials...")

Extracting materials...


In [7]:
# Run material extraction
materials_text = material_extractor.forward(input=clean_text(paper.publication_text))

# Parse into list
materials = [
    m.strip()
    for m in materials_text.replace("\n", ",").split(",")
    if m.strip()
]

print("=" * 60)
print(f"MATERIALS FOUND ({len(materials)} total)")
print("=" * 60)
for i, mat in enumerate(materials, 1):
    print(f"  {i}. {mat}")

2026/02/04 08:33:26 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


MATERIALS FOUND (6 total)
  1. Ni/SiO2
  2. Ni7Co3/SiO2
  3. Ni5Co5/SiO2
  4. Ni3Co7/SiO2
  5. Co/SiO2
  6. Ni5Co5/SiO2 e K


## Step 2: Extract Synthesis Procedures

For each material, extract the detailed synthesis procedure.

In [8]:
from llm_synthesis.transformers.synthesis_extraction.dspy_synthesis_extraction import (
    DspySynthesisExtractor,
    make_dspy_synthesis_extractor_signature,
)
from llm_synthesis.metrics.judge.general_synthesis_judge import (
    DspyGeneralSynthesisJudge,
    make_general_synthesis_judge_signature,
)

# System prompt for synthesis extraction
SYNTHESIS_SYSTEM_PROMPT = """You are a helpful assistant that extracts structured synthesis procedures from scientific papers.

IMPORTANT: For the synthesis_method field, you MUST choose from these exact values:
'PVD', 'CVD', 'arc discharge', 'ball milling', 'spray pyrolysis', 'electrospinning',
'sol-gel', 'hydrothermal', 'solvothermal', 'precipitation', 'coprecipitation', 'combustion',
'microwave-assisted', 'sonochemical', 'template-directed', 'solid-state', 'flux growth',
'float zone & Bridgman', 'arc melting & induction melting', 'spark plasma sintering',
'electrochemical deposition', 'chemical bath deposition', 'liquid-phase epitaxy', 'self-assembly',
'atomic layer deposition', 'molecular beam epitaxy', 'pulsed laser deposition', 'ion implantation',
'lithographic patterning', 'wet impregnation', 'incipient wetness impregnation', 'mechanical mixing',
'solution-based', 'mechanochemical', 'other'

For the target_compound_type field, you MUST choose from these exact values:
'metals & alloys', 'ceramics & glasses', 'polymers & soft matter', 'composites',
'semiconductors & electronic', 'nanomaterials', 'two-dimensional materials',
'framework & porous materials', 'biomaterials & biological', 'liquid materials',
'hybrid & organic-inorganic', 'functional materials & catalysts', 'energy & sustainability',
'smart & responsive materials', 'emerging & quantum materials', 'other'

If the exact method is not in the list, use the closest match or 'other'."""

# Create synthesis extractor
synthesis_sig = make_dspy_synthesis_extractor_signature(
    instructions=(
        "Extract the complete structured synthesis procedure for the specified material. "
        "Include all steps, conditions (temperature, time, atmosphere), equipment, and precursors. "
        "Be thorough and preserve all quantitative details."
    ),
)

synthesis_lm = get_llm_from_name(
    GEMINI_MODEL,
    model_kwargs={"temperature": 0.0, "max_tokens": 8000, "max_retries": 3},
    system_prompt=SYNTHESIS_SYSTEM_PROMPT,
)
synthesis_extractor = DspySynthesisExtractor(signature=synthesis_sig, lm=synthesis_lm)

# Create judge
judge_lm = get_llm_from_name(
    GEMINI_MODEL,
    model_kwargs={"temperature": 0.1, "max_tokens": 4096},
)
judge_sig = make_general_synthesis_judge_signature()
judge = DspyGeneralSynthesisJudge(signature=judge_sig, lm=judge_lm)

print("[OK] Synthesis extractor and judge initialized")

[OK] Synthesis extractor and judge initialized


In [9]:
# Extract synthesis for each material
from llm_synthesis.models.paper import SynthesisEntry

all_syntheses = []
text_for_llm = clean_text(paper.publication_text)

for i, material in enumerate(materials, 1):
    print(f"\n{'=' * 60}")
    print(f"EXTRACTING SYNTHESIS {i}/{len(materials)}: {material}")
    print("=" * 60)
    
    try:
        # Extract synthesis
        synthesis = synthesis_extractor.forward(input=(text_for_llm, material))
        
        # Evaluate
        try:
            evaluation = judge.forward(
                (text_for_llm, json.dumps(synthesis.model_dump()), material)
            )
            print(f"   [OK] Evaluation score: {evaluation.scores.overall_score}/5.0")
        except Exception as e:
            print(f"   [WARN] Judge failed: {e}")
            evaluation = None
        
        all_syntheses.append(SynthesisEntry(
            material=material,
            synthesis=synthesis,
            evaluation=evaluation,
        ))
        
        # Show synthesis summary
        print(f"\n   Target: {synthesis.target_compound}")
        print(f"   Type: {synthesis.target_compound_type}")
        print(f"   Method: {synthesis.synthesis_method}")
        print(f"   Starting materials: {len(synthesis.starting_materials)}")
        print(f"   Steps: {len(synthesis.steps)}")
        
    except Exception as e:
        print(f"   [ERROR] Extraction failed: {e}")
        all_syntheses.append(SynthesisEntry(
            material=material,
            synthesis=None,
            evaluation=None,
        ))

print(f"\n\n[OK] Extracted synthesis for {len(all_syntheses)} materials")

2026/02/04 08:33:33 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/02/04 08:33:33 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/02/04 08:33:33 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/02/04 08:33:33 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/02/04 08:33:33 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/02/04 08:33:33 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/02/04 08:33:33 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/02/04 08:33:33 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/02/04 08:33


EXTRACTING SYNTHESIS 1/6: Ni/SiO2
   [OK] Evaluation score: 4.4/5.0

   Target: Ni/SiO2
   Type: functional materials & catalysts
   Method: wet impregnation
   Starting materials: 5
   Steps: 8

EXTRACTING SYNTHESIS 2/6: Ni7Co3/SiO2
   [OK] Evaluation score: 4.3/5.0

   Target: Ni7Co3/SiO2
   Type: functional materials & catalysts
   Method: wet impregnation
   Starting materials: 4
   Steps: 8

EXTRACTING SYNTHESIS 3/6: Ni5Co5/SiO2
   [OK] Evaluation score: 4.3/5.0

   Target: Ni5Co5/SiO2
   Type: functional materials & catalysts
   Method: wet impregnation
   Starting materials: 4
   Steps: 8

EXTRACTING SYNTHESIS 4/6: Ni3Co7/SiO2
   [OK] Evaluation score: 4.0/5.0

   Target: Ni3Co7/SiO2
   Type: functional materials & catalysts
   Method: wet impregnation
   Starting materials: 4
   Steps: 8

EXTRACTING SYNTHESIS 5/6: Co/SiO2
   [OK] Evaluation score: 5.0/5.0

   Target: Co/SiO2
   Type: functional materials & catalysts
   Method: wet impregnation
   Starting materials: 3
   Steps

In [10]:
# Detailed view of first synthesis
if all_syntheses and all_syntheses[0].synthesis:
    s = all_syntheses[0].synthesis
    print("=" * 60)
    print(f"DETAILED SYNTHESIS: {all_syntheses[0].material}")
    print("=" * 60)
    
    print(f"\nStarting Materials:")
    for mat in s.starting_materials:
        amt = f"{mat.amount} {mat.unit}" if mat.amount else "N/A"
        print(f"  - {mat.name}: {amt}")
    
    print(f"\nSynthesis Steps:")
    for step in s.steps:
        print(f"  Step {step.step_number}: {step.action}")
        if step.description:
            print(f"     {step.description[:100]}..." if len(step.description) > 100 else f"     {step.description}")
        if step.conditions:
            c = step.conditions
            cond_parts = []
            if c.temperature: cond_parts.append(f"{c.temperature} {c.temp_unit or 'C'}")
            if c.duration: cond_parts.append(f"{c.duration} {c.time_unit or 'h'}")
            if c.atmosphere: cond_parts.append(c.atmosphere)
            if cond_parts:
                print(f"     Conditions: {', '.join(cond_parts)}")

DETAILED SYNTHESIS: Ni/SiO2

Starting Materials:
  - Ni(NO3)2 $ 7H2O: N/A
  - fumed SiO2: 0.9 g
  - deionized water: 50.0 mL
  - H2: N/A
  - Ar: N/A

Synthesis Steps:
  Step 1: mix
     0.9 g of fumed SiO2 was homogeneously dispersed in 50 mL of deionized water contained in a beaker
  Step 2: add
     the desired volume of metal precursor (0.5 M) was added.
  Step 3: sonicate
     After 30 min of ultrasonication
     Conditions: 30.0 min
  Step 4: stir
     the beaker was transferred to a water bath and the mixture was stirred at 30 /C14 C for 12 h
     Conditions: 30.0 C, 12.0 h
  Step 5: heat
     heated to 80 /C14 C until the solution was evaporated to dryness.
     Conditions: 80.0 C
  Step 6: dry
     The resultant catalyst precursor was dried at 80 /C14 C overnight
     Conditions: 80.0 C, 12.0 h
  Step 7: filter
     pressed and sieved into 20 e 40 mesh.
  Step 8: reduce
     Such obtained catalyst precursors were then reduced in 10% H2/Ar atmosphere (50 mL min /C0 1 ) at 55...


## Step 3: Extract Figures

Find and classify all figures in the paper, identifying quantitative plots.

In [11]:
if SKIP_FIGURES:
    print("[SKIP] Skipping figure extraction (SKIP_FIGURES=True)")
    figures = []
else:
    from llm_synthesis.transformers.figure_extraction.regex_figure_extractor import (
        FigureExtractorMarkdown,
    )
    
    print("Extracting figures from paper...")
    
    extractor = FigureExtractorMarkdown()
    all_figures = extractor.forward(paper.publication_text)
    
    # Filter for quantitative plots
    quantitative_classes = [
        "Bar plots", "Contour plot", "Graph plots",
        "Scatter plot", "Surface plot", "Vector plot",
    ]
    figures = [
        f for f in all_figures
        if f.quantitative or f.figure_class in quantitative_classes
    ]
    
    print(f"\n{'=' * 60}")
    print(f"FIGURES FOUND ({len(all_figures)} total, {len(figures)} quantitative)")
    print("=" * 60)
    for i, fig in enumerate(figures):
        print(f"  {i+1}. {fig.figure_reference or f'Figure {i}'}: {fig.figure_class}")
        if fig.alt_text:
            print(f"      Caption: {fig.alt_text[:80]}..." if len(fig.alt_text) > 80 else f"      Caption: {fig.alt_text}")

Extracting figures from paper...
Found 6 figures in the paper.


/Users/valeriegentzke/Documents/Studieren/lemat_synth/lematerial-llm-synthesis/.venv/lib/python3.11/site-packages/transformers/models/grounding_dino/processing_grounding_dino.py:94: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)


Segmented 1 subfigures.
Segmented 1 subfigures.
Segmented 15 subfigures.
Segmented 2 subfigures.
Segmented 5 subfigures.
Segmented 4 subfigures.

FIGURES FOUND (28 total, 14 quantitative)
  1. Fig. 1: Graph plots
      Caption: Image
  2. Fig. 1: Graph plots
      Caption: Image
  3. Fig. 1: Bar plots
      Caption: Image
  4. Fig. 1: Bar plots
      Caption: Image
  5. Fig. 1: Bar plots
      Caption: Image
  6. Fig. 1: Graph plots
      Caption: Image
  7. Fig. 1: Bar plots
      Caption: Image
  8. Fig. 1: Bar plots
      Caption: Image
  9. Unknown Figure: Graph plots
      Caption: Image
  10. Unknown Figure: Graph plots
      Caption: Image
  11. Fig. 4: Graph plots
      Caption: Image
  12. Fig. 4: Graph plots
      Caption: Image
  13. Fig. 4: Graph plots
      Caption: Image
  14. Fig. 4: Graph plots
      Caption: Image


## Step 4: Extract Plot Data

Use Claude VLM to extract numerical data from quantitative plots.

In [12]:
if SKIP_FIGURES or not figures:
    print("[SKIP] Skipping plot data extraction")
    plots = []
    plot_figures = []
else:
    from llm_synthesis.transformers.plot_extraction.claude_extraction.plot_data_extraction import (
        ClaudeLinePlotDataExtractor,
    )
    from llm_synthesis.models.figure import FigureInfoWithPaper
    from llm_synthesis.utils.figure_utils import clean_text_from_images
    
    print(f"Extracting data from {len(figures)} plots using Claude VLM...")
    
    plot_extractor = ClaudeLinePlotDataExtractor(model_name=CLAUDE_MODEL)
    
    plots = []
    plot_figures = []
    
    for i, fig in enumerate(figures):
        print(f"\n  Processing figure {i+1}/{len(figures)}: {fig.figure_reference or f'Figure {i}'}")
        
        fig_with_paper = FigureInfoWithPaper(
            base64_data=fig.base64_data,
            alt_text=fig.alt_text,
            position=fig.position,
            context_before=fig.context_before,
            context_after=fig.context_after,
            figure_reference=fig.figure_reference,
            figure_class=fig.figure_class,
            quantitative=fig.quantitative,
            paper_text=clean_text_from_images(paper.publication_text),
            si_text=paper.si_text,
        )
        
        try:
            plot_data = plot_extractor.forward(fig_with_paper)
            if plot_data and plot_data.name_to_coordinates:
                plots.append(plot_data)
                plot_figures.append(fig)
                print(f"    [OK] Extracted {len(plot_data.name_to_coordinates)} series")
                print(f"       Series: {list(plot_data.name_to_coordinates.keys())}")
            else:
                print(f"    [WARN] No data extracted")
        except Exception as e:
            print(f"    [ERROR] Failed: {e}")
    
    print(f"\n[OK] Extracted data from {len(plots)} plots")

Extracting data from 14 plots using Claude VLM...

  Processing figure 1/14: Fig. 1
    [OK] Extracted 3 series
       Series: ['NiSiO₂', 'Ni₂Co₂SiO₂', 'CoSiO₂']

  Processing figure 2/14: Fig. 1
    [OK] Extracted 5 series
       Series: ['NiSiO₂', 'Ni₃Co₇/SiO₂', 'Ni₅Co₅/SiO₂', 'Ni₇Co₃/SiO₂', 'CoSiO₂']

  Processing figure 3/14: Fig. 1
    [WARN] No data extracted

  Processing figure 4/14: Fig. 1
    [OK] Extracted 2 series
       Series: ['Experiment', 'Theory']

  Processing figure 5/14: Fig. 1
    [WARN] No data extracted

  Processing figure 6/14: Fig. 1
    [OK] Extracted 3 series
       Series: ['NiSiO₂', 'Ni₂Co₃SiO₂', 'Co₃SiO₂']

  Processing figure 7/14: Fig. 1
    [OK] Extracted 2 series
       Series: ['Experiment', 'Theory']

  Processing figure 8/14: Fig. 1
    [OK] Extracted 3 series
       Series: ['Blue line', 'Red line', 'Green line']

  Processing figure 9/14: Unknown Figure
    [OK] Extracted 3 series
       Series: ['CoSiO₂', 'Ni₂Co₂SiO₂', 'NiSiO₂']

  Processing f

In [13]:
# Show plot details
if plots:
    print("=" * 60)
    print("EXTRACTED PLOT DATA SUMMARY")
    print("=" * 60)
    for i, (plot, fig) in enumerate(zip(plots, plot_figures)):
        print(f"\nPlot {i}: {fig.figure_reference or 'N/A'}")
        print(f"  Title: {plot.title or 'N/A'}")
        print(f"  X-axis: {plot.x_axis_label} [{plot.x_axis_unit}]")
        print(f"  Y-axis: {plot.y_left_axis_label} [{plot.y_left_axis_unit}]")
        print(f"  Series ({len(plot.name_to_coordinates)}):")
        for series_name, coords in plot.name_to_coordinates.items():
            print(f"    - {series_name}: {len(coords)} points")

EXTRACTED PLOT DATA SUMMARY

Plot 0: Fig. 1
  Title: N/A
  X-axis: 1000/T [K⁻¹]
  Y-axis:  []
  Series (3):
    - NiSiO₂: 4 points
    - Ni₂Co₂SiO₂: 4 points
    - CoSiO₂: 4 points

Plot 1: Fig. 1
  Title: N/A
  X-axis: Temperature [°C]
  Y-axis: NH₃ Conversion [%]
  Series (5):
    - NiSiO₂: 6 points
    - Ni₃Co₇/SiO₂: 6 points
    - Ni₅Co₅/SiO₂: 6 points
    - Ni₇Co₃/SiO₂: 6 points
    - CoSiO₂: 6 points

Plot 2: Fig. 1
  Title: Comparison of Experimental and Theoretical Results
  X-axis: Time [s]
  Y-axis: Amplitude [V]
  Series (2):
    - Experiment: 11 points
    - Theory: 11 points

Plot 3: Fig. 1
  Title: N/A
  X-axis: P / P° []
  Y-axis: Quantity adsorbed [cm³ g⁻¹]
  Series (3):
    - NiSiO₂: 12 points
    - Ni₂Co₃SiO₂: 12 points
    - Co₃SiO₂: 12 points

Plot 4: Fig. 1
  Title: Comparison of Experimental and Theoretical Results
  X-axis: Time [s]
  Y-axis: Amplitude [V]
  Series (2):
    - Experiment: 11 points
    - Theory: 11 points

Plot 5: Fig. 1
  Title: Polynomial Functi

## Step 5: Configure Plot Filtering (Optional)

Configure which plots should be included in performance linking based on axis characteristics.

In [14]:
from llm_synthesis.config.plot_filter_config import PlotFilterConfig
from llm_synthesis.transformers.performance_linking.plot_filter import PlotFilter

# ==============================================================================
# CONFIGURE PLOT FILTERING
# ==============================================================================

# Option 1: Default catalysis config (temperature x-axis, conversion y-axis)
filter_config = PlotFilterConfig.for_catalysis()

# Option 2: Electrochemistry config (potential x-axis, current/capacitance y-axis)
# filter_config = PlotFilterConfig.for_electrochemistry()

# Option 3: No filtering (link all plots)
# filter_config = PlotFilterConfig.no_filter()

# Option 4: Custom config
# filter_config = PlotFilterConfig(
#     x_axis_labels=["time", "t"],
#     x_axis_units=["h", "min", "s"],
#     y_axis_keywords=["conversion", "yield"],
#     y_axis_units=["%"],
# )

plot_filter = PlotFilter(filter_config)

print("Plot Filter Configuration:")
print(f"   X-axis labels: {filter_config.x_axis_labels}")
print(f"   X-axis units: {filter_config.x_axis_units}")
print(f"   Y-axis keywords: {filter_config.y_axis_keywords}")
print(f"   Y-axis units: {filter_config.y_axis_units}")
print(f"   Filter X-axis: {filter_config.filter_x_axis}")
print(f"   Filter Y-axis: {filter_config.filter_y_axis}")

Plot Filter Configuration:
   X-axis labels: ['t', 'temp', 'temperature']
   X-axis units: ['°c', '°k', '°f', 'ºc', 'ºk']
   Y-axis keywords: ['conversion', 'yield', 'selectivity', 'activity']
   Y-axis units: ['%', 'percent']
   Filter X-axis: True
   Filter Y-axis: True


In [15]:
# Apply filtering
if plots:
    relevant_plots, skip_counts = plot_filter.filter_plots(plots, log_skipped=True)
    
    print(f"\n{'=' * 60}")
    print("PLOT FILTERING RESULTS")
    print("=" * 60)
    print(f"  Total plots: {len(plots)}")
    print(f"  Relevant plots: {len(relevant_plots)}")
    print(f"  Skipped (not relevant x-axis): {skip_counts.get('not_relevant_x', 0)}")
    print(f"  Skipped (not relevant y-axis): {skip_counts.get('not_relevant_y', 0)}")
    print(f"  Skipped (no series): {skip_counts.get('no_series', 0)}")
    
    print(f"\n  Relevant plots for linking:")
    for idx, plot in relevant_plots:
        print(f"    Plot {idx}: {plot.title or 'N/A'} ({len(plot.name_to_coordinates)} series)")
else:
    print("[SKIP] No plots to filter")
    relevant_plots = []


PLOT FILTERING RESULTS
  Total plots: 12
  Relevant plots: 2
  Skipped (not relevant x-axis): 7
  Skipped (not relevant y-axis): 3
  Skipped (no series): 0

  Relevant plots for linking:
    Plot 1: N/A (5 series)
    Plot 10: N/A (2 series)


## Step 6: Link Series to Materials

Use LLM to match plot series names to extracted materials.

In [16]:
if SKIP_FIGURES or not relevant_plots:
    print("[SKIP] Skipping performance linking")
    plot_mappings = []
    linking_stats = None
else:
    import dspy
    from llm_synthesis.transformers.performance_linking.series_material_linker import (
        SeriesMaterialLinker,
    )
    from llm_synthesis.transformers.performance_linking.base import LinkingInput
    from llm_synthesis.models.performance import PlotMaterialMapping
    
    print(f"Linking plot series to {len(materials)} materials...")
    
    # Initialize linker
    gemini_key = os.getenv("GEMINI_API_KEY", "")
    linker_lm = dspy.LM(
        f"gemini/{LINKER_MODEL}",
        temperature=0.0,
        max_tokens=8000,
        api_key=gemini_key,
    )
    series_linker = SeriesMaterialLinker(lm=linker_lm)
    
    plot_mappings = []
    
    for idx, plot in relevant_plots:
        fig = plot_figures[idx]
        series_names = list(plot.name_to_coordinates.keys())
        
        print(f"\n  Linking plot {idx}: '{plot.title or 'N/A'}' ({len(series_names)} series)")
        print(f"    Series: {series_names}")
        
        context = f"{fig.context_before} {fig.context_after}"
        plot_meta = {
            "title": plot.title,
            "x_axis_label": plot.x_axis_label,
            "x_axis_unit": plot.x_axis_unit,
            "y_left_axis_label": plot.y_left_axis_label,
            "y_left_axis_unit": plot.y_left_axis_unit,
        }
        
        # Call linker
        linking_input = LinkingInput(
            materials=materials,
            series_names=series_names,
            context=context,
            plot_metadata=plot_meta,
        )
        validated_mappings = series_linker.forward(linking_input)
        
        # Determine unmatched
        matched_series = {m.series_name for m in validated_mappings}
        unmatched = [s for s in series_names if s not in matched_series]
        
        plot_mappings.append(PlotMaterialMapping(
            plot_index=idx,
            figure_reference=fig.figure_reference,
            mappings=validated_mappings,
            unmatched_series=unmatched,
        ))
        
        print(f"    [OK] Matched: {len(validated_mappings)}")
        for m in validated_mappings:
            print(f"       '{m.series_name}' -> '{m.material_name}' ({m.confidence})")
        if unmatched:
            print(f"    [WARN] Unmatched: {unmatched}")
    
    print(f"\n[OK] Linking complete")

Linking plot series to 6 materials...

  Linking plot 1: 'N/A' (5 series)
    Series: ['NiSiO₂', 'Ni₃Co₇/SiO₂', 'Ni₅Co₅/SiO₂', 'Ni₇Co₃/SiO₂', 'CoSiO₂']
    [OK] Matched: 5
       'NiSiO₂' -> 'Ni/SiO2' (high)
       'Ni₃Co₇/SiO₂' -> 'Ni3Co7/SiO2' (high)
       'Ni₅Co₅/SiO₂' -> 'Ni5Co5/SiO2' (high)
       'Ni₇Co₃/SiO₂' -> 'Ni7Co3/SiO2' (high)
       'CoSiO₂' -> 'Co/SiO2' (high)

  Linking plot 10: 'N/A' (2 series)
    Series: ['Ni₂Co/SiO₂-KOH', 'Ni₂Co/SiO₂']
    [OK] Matched: 2
       'Ni₂Co/SiO₂-KOH' -> 'Ni5Co5/SiO2 e K' (high)
       'Ni₂Co/SiO₂' -> 'Ni5Co5/SiO2' (high)

[OK] Linking complete


## Step 7: Aggregate Performance Data

Combine all performance data per material.

In [17]:
from llm_synthesis.utils.performance_utils import aggregate_all_materials_performance

if plot_mappings and plots:
    performance_data = aggregate_all_materials_performance(materials, plot_mappings, plots)
    
    print("=" * 60)
    print("PER-MATERIAL PERFORMANCE SUMMARY")
    print("=" * 60)
    
    for mat in materials:
        if mat in performance_data:
            perf = performance_data[mat]
            print(f"\n{mat}: {len(perf.plot_data)} performance entries")
            for entry in perf.plot_data:
                print(
                    f"  - {entry.plot_title or 'N/A'} / series '{entry.series_name}' "
                    f"({entry.y_axis_label} [{entry.y_axis_unit}]), "
                    f"{len(entry.coordinates)} points, confidence: {entry.confidence}"
                )
        else:
            print(f"\n{mat}: (no performance data linked)")
else:
    performance_data = {}
    print("[SKIP] No performance data to aggregate")

PER-MATERIAL PERFORMANCE SUMMARY

Ni/SiO2: 1 performance entries
  - N/A / series 'NiSiO₂' (NH₃ Conversion [%]), 6 points, confidence: high

Ni7Co3/SiO2: 1 performance entries
  - N/A / series 'Ni₇Co₃/SiO₂' (NH₃ Conversion [%]), 6 points, confidence: high

Ni5Co5/SiO2: 2 performance entries
  - N/A / series 'Ni₅Co₅/SiO₂' (NH₃ Conversion [%]), 6 points, confidence: high
  - N/A / series 'Ni₂Co/SiO₂' (NH₃ Conversion [%]), 6 points, confidence: high

Ni3Co7/SiO2: 1 performance entries
  - N/A / series 'Ni₃Co₇/SiO₂' (NH₃ Conversion [%]), 6 points, confidence: high

Co/SiO2: 1 performance entries
  - N/A / series 'CoSiO₂' (NH₃ Conversion [%]), 6 points, confidence: high

Ni5Co5/SiO2 e K: 1 performance entries
  - N/A / series 'Ni₂Co/SiO₂-KOH' (NH₃ Conversion [%]), 6 points, confidence: high


## Step 8: Build Final Results

In [18]:
# Combine synthesis + performance
final_results = []

for entry in all_syntheses:
    result = {
        "material": entry.material,
        "synthesis": entry.synthesis.model_dump() if entry.synthesis else None,
        "evaluation": entry.evaluation.model_dump() if entry.evaluation else None,
        "performance": (
            performance_data[entry.material].model_dump()
            if entry.material in performance_data
            else None
        ),
    }
    final_results.append(result)

# Summary
materials_with_perf = [m for m in materials if m in performance_data]
materials_without_perf = [m for m in materials if m not in performance_data]

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"  Paper: {paper.name}")
print(f"  Total materials: {len(materials)}")
print(f"  Materials with synthesis: {sum(1 for r in final_results if r['synthesis'])}")
print(f"  Materials with performance: {len(materials_with_perf)}")
print(f"  Materials without performance: {len(materials_without_perf)}")
if plots:
    print(f"  Total plots extracted: {len(plots)}")
    print(f"  Plots linked: {len(plot_mappings)}")

FINAL SUMMARY
  Paper: 04_Wu_2020_Ammonia
  Total materials: 6
  Materials with synthesis: 6
  Materials with performance: 6
  Materials without performance: 0
  Total plots extracted: 12
  Plots linked: 2


In [19]:
# Show one complete result
if final_results:
    # Find a result with both synthesis and performance
    example = next((r for r in final_results if r['synthesis'] and r['performance']), final_results[0])
    
    print("=" * 60)
    print(f"EXAMPLE RESULT: {example['material']}")
    print("=" * 60)
    print(json.dumps(example, indent=2, default=str)[:3000])
    if len(json.dumps(example, indent=2)) > 3000:
        print("... (truncated)")

EXAMPLE RESULT: Ni/SiO2
{
  "material": "Ni/SiO2",
  "synthesis": {
    "target_compound": "Ni/SiO2",
    "target_compound_type": "functional materials & catalysts",
    "synthesis_method": "wet impregnation",
    "starting_materials": [
      {
        "vendor": "Shanghai Macklin Biochemical Co., Ltd.",
        "name": "Ni(NO3)2 $ 7H2O",
        "amount": null,
        "unit": null,
        "purity": "AR, 99%"
      },
      {
        "vendor": "Shanghai Macklin Biochemical Co., Ltd.",
        "name": "fumed SiO2",
        "amount": 0.9,
        "unit": "g",
        "purity": null
      },
      {
        "vendor": null,
        "name": "deionized water",
        "amount": 50.0,
        "unit": "mL",
        "purity": null
      },
      {
        "vendor": null,
        "name": "H2",
        "amount": null,
        "unit": null,
        "purity": "10%"
      },
      {
        "vendor": null,
        "name": "Ar",
        "amount": null,
        "unit": null,
        "purity": "90%"


## Step 9: Save Results

In [20]:
import os
from llm_synthesis.utils.performance_utils import sanitize_filename

# Create output directory
paper_dir = os.path.join(OUTPUT_DIR, paper.id)
os.makedirs(paper_dir, exist_ok=True)

# Save individual material files
for result in final_results:
    mat_name = sanitize_filename(result["material"])
    mat_path = os.path.join(paper_dir, f"{mat_name}.json")
    with open(mat_path, "w") as f:
        json.dump(result, f, indent=2, default=str)

# Save plot mappings
if plot_mappings:
    mappings_path = os.path.join(paper_dir, "performance_mappings.json")
    with open(mappings_path, "w") as f:
        json.dump([m.model_dump() for m in plot_mappings], f, indent=2)

# Save summary
summary = {
    "paper_id": paper.id,
    "paper_name": paper.name,
    "total_materials": len(materials),
    "materials_with_performance": len(materials_with_perf),
    "materials_without_performance": len(materials_without_perf),
    "materials_list": materials,
    "materials_with_performance_list": materials_with_perf,
    "materials_without_performance_list": materials_without_perf,
    "total_plots_extracted": len(plots) if plots else 0,
    "plots_linked": len(plot_mappings),
}

summary_path = os.path.join(paper_dir, "summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"[OK] Results saved to: {paper_dir}/")
print(f"   - {len(final_results)} material files")
print(f"   - performance_mappings.json")
print(f"   - summary.json")

[OK] Results saved to: results/04_Wu_2020_Ammonia/
   - 6 material files
   - performance_mappings.json
   - summary.json


---

## Done!

You have successfully extracted:
- **Materials** from the paper
- **Synthesis procedures** for each material
- **Performance data** from plots, linked to specific materials

Check the output directory for the saved results.